In [ ]:
import itertools
import math

import cirq
import cirq_google
import numpy as np

# Google Quantum AI Challenge Problem

Your code may import from cirq, cirq_google, their dependencies, and python's standard library. Other imports are not allowed.

**Please upload your solutions to [this form](https://docs.google.com/forms/d/e/1FAIpQLScw93lwRb3OihfaHgpobs8TO6nXijwvkRVYjWPpNg7n92FjIA/viewform?usp=publish-editor).** See the form for further instructions.

Consider an $N$-qubit, weight $n$ Dicke state, defined by
$$|D^{(N)}_n\rangle = \begin{pmatrix} N \\ n \end{pmatrix}^{-1/2} \sum_k \mathcal{P}_k \left( |0\rangle^{\otimes(N-n)} \otimes |1\rangle^{\otimes n}\right),$$
where the sum is over all distinct permutations $\mathcal{P}_k$. For $n=1$, this is the $W$ state, given by
$$|W\rangle = \frac{1}{\sqrt N} \left( |100\ldots 0 \rangle + |010\ldots 0 \rangle + \ldots + |00\ldots 01 \rangle \right). $$

Your task is to complete the following function

In [ ]:
def create_dicke_state(
    num_qubits: int,
    num_excitations: int,
    cz_pauli_error_rate: dict[
        tuple[cirq.GridQubit, cirq.GridQubit], tuple[float, float]
    ],
    sq_pauli_error_rate: dict[cirq.GridQubit, tuple[float, float]],
    zero_errors: dict[cirq.GridQubit, tuple[float, float]],
    one_errors: dict[cirq.GridQubit, tuple[float, float]],
) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
    """Create a Cirq circuit that produces a Dicke state.

    Args:
        num_qubits: The number, N, of qubits in the Dicke state.
        num_excitations: The number, n, of 1s in the state.
        cz_pauli_error_rate: Measured CZ Pauli error rates and their statistical uncertainties.
        sq_pauli_error_rate: Measured single-qubit Pauli error rates and their statistical uncertainties.
        zero_errors: Measured parallel 0 -> 1 readout error rates and their statistical uncertainties.
        one_errors: Measured 1 -> 0 readout error rates and their statistical uncertainties.

    Returns:
        A Cirq circuit to produce Dicke state and a set indicating which qubits the resulting state lives on.

    Raises:
        NotImplementedError: The requested (N, n) combination is not implemented.
    """


For small $N$, you may check that your circuit produces the correct state using the following function:

In [ ]:
def create_dicke_statevector(num_qubits: int, num_excitations: int) -> np.ndarray:
    psi = np.zeros(2**num_qubits)
    normalization = 1 / np.sqrt(math.comb(num_qubits, num_excitations))
    qubits = range(num_qubits)
    for excited_qubits in itertools.combinations(qubits, num_excitations):
        idx = sum(2**qubit_number for qubit_number in excited_qubits)
        psi[idx] = normalization
    return psi


def test_circuit(circuit: cirq.Circuit, num_excitations: int, qubits: set[cirq.GridQubit], sampler = cirq.Simulator()):
    all_qubits = circuit.all_qubits()
    ancilla_qubits = all_qubits.difference(qubits)
    ordered_qubits = sorted(qubits) + sorted(ancilla_qubits)
    keep_indices = range(qubits)
    num_qubits = len(qubits)
    psi_circuit_full = sampler.simulate(circuit, qubit_order=ordered_qubits).final_state_vector
    psi_circuit = cirq.sub_state_vector(
        state_vector=psi_circuit_full,
        keep_indices=keep_indices
    ) if len(all_qubits) > len(qubits) else psi_circuit_full
    assert np.isclose(np.abs(psi_circuit @ create_dicke_statevector(num_qubits, num_excitations)) ** 2, 1.0)

### Requirements:
* The resulting circuit is allowed to contain only
  * CZ gates between pairs of qubits included in `cz_pauli_error_rate`, which will be a subset (and most) of the connected pairs in the [spec sheet](https://quantumai.google/static/site-assets/downloads/willow-spec-sheet.pdf).
  * Arbitrary single-qubit unitaries (`cirq.PhasedXZGate`) on qubits included in `sq_pauli_error_rate`, which will be most of the qubits in the spec sheet.
  * Optionally, mid-circuit reset gates.
* Classically-conditioned gates are not supported.
* In the absence of noise, the circuit must either produce perfect Dicke states of the requested $N$ and $n$ or throw a `NotImplementedError`.
* If your circuit construction code takes longer than a minute to return a circuit, that is counted the same as returning a `NotImplementedError`.
* Any circuits containing more than 1750 moments will not be run.

# Scoring

We prepare the Dicke state on actual quantum hardware and measure it in both the Z basis and the X basis. We use the following `overall_score` function to compare against the expected distributions.

In [ ]:
def check_z_bitstring_distribution(
    bitstrings: np.ndarray, num_excitations: int
) -> float:
    """Measure how close the observed Z-basis distribution of bitstrings is to
    the ideal one. The lower the score, the better.

    Args:
        bitstrings: An array of shape (n_shots, n_qubits) containing the n_shots Z-basis bitstrings.
        num_excitations: The weight, n, of the Dicke state.

    Returns:
        A measure of how close the distribution is to the ideal one.
    """
    # first, let's see how many bitstrings have one excitation
    n_shots, n_qubits = bitstrings.shape
    measured_excitations = np.sum(bitstrings, axis=1)
    fraction_in_subspace = np.mean(measured_excitations == num_excitations)

    # now, let's check how uniform the distribution is over states with n excitations
    chi2 = 0
    ideal_prob = 1 / math.comb(n_qubits, num_excitations)
    for indices in itertools.combinations(range(n_qubits), num_excitations):
        ideal_state = np.zeros(n_qubits)
        ideal_state[np.array(indices)] = 1
        match_count = np.sum(np.all(bitstrings == ideal_state, axis=1))
        expected_count = n_shots * fraction_in_subspace * ideal_prob
        chi2 += (match_count - expected_count) ** 2 / expected_count

    return ideal_prob * chi2 / fraction_in_subspace


def get_correlator_average(bitstrings: np.ndarray, correlator_size: int) -> float:
    """Get the average of all weight-correlator_size correlators in `bitstrings`.

    Args:
        bitstrings: The input bitstrings, shape (n_shots, n_qubits).
        correlator_size: Compute all correlators of this weight.

    Returns:
        The average of all weight-k correlators.
    """
    correlator_sum = 0.0
    n_shots, n_qubits = bitstrings.shape
    for indices in itertools.combinations(range(n_qubits), correlator_size):
        correlator_sum += 1 - 2 * np.mean(np.sum(bitstrings[:, indices], axis=1) % 2)
    return correlator_sum / math.comb(n_qubits, correlator_size)


def check_x_bitstring_distribution(
    bitstrings: np.ndarray, num_excitations: int, max_correlator_size: int
) -> float:
    """Check expected correlations in the X basis. The lower the score, the better.

    Args:
        bitstrings: An array of shape (n_shots, n_qubits) containing the n_shots X-basis bitstrings.
        max_correlator_size: The maximum weight of correlators to consider.
        num_excitations: The weight, n, of the Dicke state.

    Returns:
        A measure of how the correlations in the X-basis measurements compare to expectation.
    """
    comb = lambda n, k: math.comb(n, k) if (k >= 0 and n >= k) else 0
    n_shots, n_qubits = bitstrings.shape
    score = 0.0
    for correlator_size in range(1, max_correlator_size + 1):
        expected_correlator = (
            comb(correlator_size, correlator_size // 2)
            * comb(n_qubits - correlator_size, num_excitations - correlator_size // 2)
            / comb(n_qubits, num_excitations)
            if correlator_size % 2 == 0
            else 0.0
        )
        measured_correlator = get_correlator_average(bitstrings, correlator_size)
        score += (
            (expected_correlator - measured_correlator) ** 2 / expected_correlator**2
            if expected_correlator != 0
            else (expected_correlator - measured_correlator) ** 2
        )

    return score


def overall_score(
    z_bitstrings: np.ndarray,
    x_bitstrings: np.ndarray,
    num_excitations: int,
    max_correlator_size: int,
) -> float:
    """An overall score that checks how the Z and X basis bitstrings compare to their ideal
    distributions. The higher the score, the better.

    Args:
        z_bitstrings: The bitstrings measured in the Z basis
        x_bitstrings: The bitstrings measured in the X basis
        num_excitations: The weight, n, of the Dicke state.
        max_correlator_size: The maximum weight of correlators to consider

    Returns:
        A measure of how similar the bitstrings are to their ideal distributions.
    """
    return 1 / (
        check_z_bitstring_distribution(z_bitstrings, num_excitations)
        + check_x_bitstring_distribution(
            x_bitstrings, num_excitations, max_correlator_size
        )
    )


### Judging criteria:
* We will run circuits generated by your code for various choices of `num_qubits` and `num_excitations` on real quantum hardware, measuring in the Z basis as well as in the X basis, each for 10,000 shots.
* For each choice of `num_qubits` and `num_excitations`, we use the `overall_score` function to score the performance. We then sum this over the choices of `num_qubits`.
* The solution with the highest score wins.

### Things to consider:
* It is permitted to use AI to answer these questions, but please include any prompts you gave to the AI in your submission.
* This will be run on real quantum hardware, so techniques like dynamical decoupling are important.


You may use the following functions to generate fake error rates to test your algorithm.

In [ ]:
def pick_single_qubit_dropouts(
    dropout_rate: float,
    rng: np.random.Generator = np.random.default_rng()
) -> set[cirq.GridQubit]:
    """Pick some qubits to be broken.

    Args:
        dropout_rate: The probability that a qubit is broken.
        rng: The pseudo-random number generator

    Returns:
        The broken qubits.
    """
    all_qubits = np.array(list(cirq_google.Willow105.metadata.qubit_set))
    is_dropout = rng.random(len(all_qubits)) < dropout_rate
    return set(all_qubits[is_dropout])


def generate_qubit_error_rates(
    median: float,
    spread: float,
    dropouts: set[cirq.GridQubit],
    uncertainty: float,
    rng: np.random.Generator = np.random.default_rng(),
) -> dict[cirq.GridQubit, tuple[float, float]]:
    """Generate fake single-qubit error rates.

    Args:
        median: The median error rate.
        spread: The spread of the lognormal distribution
        dropouts: Qubits to exclude
        uncertainty: The uncertainty of the measured error rates.
        rng: The pseudo-random number generator

    Returns:
        A dictionary from qubits to the measured error rate and statistical uncertainty.
    """

    all_qubits = np.array(list(cirq_google.Willow105.metadata.qubit_set))
    is_working = np.array([q not in dropouts for q in all_qubits])
    working_qubits = all_qubits[is_working]
    error_rate = rng.lognormal(np.log(median), spread, len(working_qubits))
    return dict(
        zip(working_qubits, zip(error_rate, uncertainty * np.ones(len(working_qubits))))
    )


def generate_two_qubit_pauli_error_rates(
    median: float = 2.5e-3,
    spread: float = 0.5,
    dropout_rate: float = 0.01,
    uncertainty: float = 5e-4,
    rng: np.random.Generator = np.random.default_rng(),
    single_qubit_dropouts: set[cirq.GridQubit] = None,
) -> dict[tuple[cirq.GridQubit, cirq.GridQubit], tuple[float, float]]:
    """Generate fake two-qubit error rates.

    Args:
        median: The median error rate.
        spread: The spread of the lognormal distribution
        dropout_rate: The probability of a broken coupler.
        uncertainty: The uncertainty of the measured error rates.
        rng: The pseudo-random number generator
        single_qubit_dropouts: The single-qubit dropouts to skip.

    Returns:
        A dictionary from qubits to the measured error rate and statistical uncertainty.
    """
    if single_qubit_dropouts is None:
        single_qubit_dropouts = set()

    all_pairs = np.array(list(cirq_google.Willow105.metadata.qubit_pairs))
    no_qubit_dropouts = np.array(
        [np.all([q not in single_qubit_dropouts for q in pair]) for pair in all_pairs]
    )
    is_working = (rng.random(len(all_pairs)) > dropout_rate) * no_qubit_dropouts
    working_pairs = [tuple(pair) for pair in all_pairs[is_working]]
    error_rate = rng.lognormal(np.log(median), spread, len(working_pairs))
    return dict(
        zip(working_pairs, zip(error_rate, uncertainty * np.ones(len(working_pairs))))
    )


In [ ]:
single_qubit_dropouts = pick_single_qubit_dropouts(0.01)
sq_pauli_error_rate = generate_qubit_error_rates(4e-4, 0.5, single_qubit_dropouts, 2e-4)
cz_pauli_error_rate = generate_two_qubit_pauli_error_rates(
    single_qubit_dropouts=single_qubit_dropouts
)
zero_errors = generate_qubit_error_rates(5e-3, 0.5, single_qubit_dropouts, 2e-4)
one_errors = generate_qubit_error_rates(1e-2, 0.5, single_qubit_dropouts, 3e-4)

# Bonus question:
Now suppose we allow for adaptive circuits, i.e. those with mid-circuit measurements and subsequent gates conditioned on the outcomes of those measurements. See if you can construct shallower circuits to prepare Dicke states given this capability. We will not run these circuits on the hardware.

For all $(N, n)$, with $2 \leq N \leq 15$, and $1 \leq n \leq \lfloor N/2 \rfloor$, we will compare the depth of the circuits produced by the various submissions and assign a point to the one with the shallowest depth, measured by the number of moments in the circuit.

Specifically, your task is to complete the following function:

In [ ]:
def create_dicke_state_adaptive(
    num_qubits: int, num_excitations: int
) -> tuple[cirq.Circuit, set[cirq.GridQubit]]:
    """Create a Cirq circuit that produces a Dicke state, allowing for adaptive circuits.

    Args:
        num_qubits: The number, N, of qubits in the Dicke state.
        num_excitations: The number, n, of 1s in the state.

    Returns:
        A Cirq circuit to produce Dicke state and a set indicating which qubits the resulting state lives on.

    Raises:
        NotImplementedError: The requested (N, n) combination is not implemented.
    """

## Scoring

We will use the following function to score submissions:

In [ ]:
def score_solution() -> float:
    score = 0
    for num_qubits in range(2, 21):
        for num_excitations in range(1, num_qubits//2+1):
            try:
                circuit, qubits = create_dicke_state_adaptive(num_qubits, num_excitations)
            except NotImplementedError:
                continue
            validate_circuit(circuit, num_excitations, qubits) # checks that the circuit is valid
            score += num_qubits * num_excitations / len(circuit)
    return score

Notes and constraints:
* We will verify classically that the circuits produce the desired Dicke state up to a global phase.
* Using adaptive circuits is not required, so you may reuse your solution from the previous question.
* Although we do not provide error rates for this question, it is still necessary to respect the connectivity of the [Willow device](https://quantumai.google/static/site-assets/downloads/willow-spec-sheet.pdf), i.e. you can only use qubits on that device and CZ gates between adjacent qubits on the device. You may use `cirq_google.Willow105.validate_circuit(circuit)` to validate that your circuits are correctly routed to such a device.
* On real hardware, qubits along a diagonal such that row # + column # = const share a readout line and measuring one qubit dephases the other qubits on that readout line. Therefore, mid-circuit measurements of some but not all of the qubits on a readout line are not allowed. (This constraint is not checked by `cirq_google.Willow105.validate_circuit(circuit)`.)
* As before, the code must execute in under a minute for each $(N, n)$.